In [3]:
!pip install numpy pandas scikit-learn matplotlib seaborn
!pip install torch transformers datasets gradio
!pip install nltk

  Using cached torch-2.11.0-cp310-cp310-win_amd64.whl.metadata (29 kB)
  Using cached transformers-5.5.3-py3-none-any.whl.metadata (32 kB)
  Using cached datasets-4.8.4-py3-none-any.whl.metadata (19 kB)
  Using cached gradio-6.11.0-py3-none-any.whl.metadata (17 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached fsspec-2026.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached huggingface_hub-1.10.1-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached hf_xet-1.4.3-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached pyarrow-23.0.1-cp310-cp310-win_amd64.whl.metadata (3.1 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached multiprocess-0.70.19-py310-none-any

In [2]:
import pandas as pd
import numpy as np
import re
import pickle
from collections import Counter
from sklearn.model_selection import train_test_split

In [4]:
import re, math, pickle, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

In [5]:

import nltk

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import torch
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\arshi/nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [9]:
df = pd.read_csv("data/train-balanced-sarcasm.csv") 

df = df.rename(columns={
    "comment": "text",
    "label": "label"
})

df = df[['text', 'label']]
df.dropna(inplace=True)

df.head()

,text,label
0,NC and NH.,0
1,You do know west teams play against west teams...,0
2,"They were underdogs earlier today, but since G...",0
3,"This meme isn't funny none of the ""new york ni...",0
4,I could use one of those tools.,0


In [10]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)        
    text = re.sub(r'[^a-zA-Z\s]', '', text)    
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['clean_text'] = df['text'].apply(clean_text)
df.head()

,text,label,clean_text
0,NC and NH.,0,nc nh
1,You do know west teams play against west teams...,0,know west teams play west teams east teams right
2,"They were underdogs earlier today, but since G...",0,underdogs earlier today since gronks announcem...
3,"This meme isn't funny none of the ""new york ni...",0,meme isnt funny none new york nigga ones
4,I could use one of those tools.,0,could use one tools


In [11]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['clean_text'],
    df['label'],
    test_size=0.1,
    random_state=42
)

In [12]:
from collections import Counter

def tokenize(text):
    return text.split()

all_tokens = []
for text in train_texts:
    all_tokens.extend(tokenize(text))

vocab = Counter(all_tokens)
vocab = {word: i+1 for i, (word, _) in enumerate(vocab.items())}

In [13]:
def encode(text, vocab):
    return [vocab.get(word, 0) for word in text.split()]

train_sequences = [encode(text, vocab) for text in train_texts]
val_sequences = [encode(text, vocab) for text in val_texts]

In [14]:
from torch.nn.utils.rnn import pad_sequence

train_sequences = [torch.tensor(seq) for seq in train_sequences]
val_sequences = [torch.tensor(seq) for seq in val_sequences]

train_padded = pad_sequence(train_sequences, batch_first=True)
val_padded = pad_sequence(val_sequences, batch_first=True)

In [15]:
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = torch.tensor(labels.values)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

In [16]:
train_dataset = TextDataset(train_padded, train_labels)
val_dataset = TextDataset(val_padded, val_labels)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [17]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size+1, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        out = self.fc(hidden[-1])
        return out

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTMModel(len(vocab)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [19]:
for epoch in range(5):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

KeyboardInterrupt: 